# USD/NGN 2-Hour Prediction System — Training Pipeline

**Pipeline:**
1. Load Quidax USDT/NGN trade data (2-hour bars)
2. Fetch external data (Brent, DXY, VIX, regional FX, BTC/USD, USDNGN official)
3. Merge and align all datasets
4. Compute BTC premium and cross-venue features
5. Engineer 42 features across 6 categories
6. Apply recency weighting and blended target
7. Train XGBoost + LightGBM + Ridge with Optuna tuning and walk-forward CV
8. Threshold sweep for production calibration
9. Export production artifacts

**Additional Logic:**
- Recency-weighted training (half-life 180 days) to address regime decay
- Blended target (Quidax + official rate) to reduce circularity
- USDNGN=X official rate added as cross-venue feature
- Optuna optimizes at production threshold (0.30%) not 0.15%
- All variable references use `public_features` consistently

---
## Step 0: Install Dependencies

In [1]:
!pip install -q yfinance xgboost lightgbm scikit-learn pandas numpy plotly optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 413.9/413.9 kB 10.7 MB/s eta 0:00:00


In [2]:
import pandas as pd
import numpy as np
import json
import warnings
import optuna
from datetime import datetime, timedelta
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

warnings.filterwarnings('ignore')
optuna.logging.set_verbosity(optuna.logging.WARNING)
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.6f}'.format)

print('All imports OK')

All imports OK


---
## Step 1: Load Quidax Proprietary Data

BigQuery CSV export of 2-hour USDT/NGN bars.

In [3]:
from google.colab import files

uploaded = files.upload()
filename = list(uploaded.keys())[0]
print(f'Uploaded: {filename}')

Saving bquxjob_46cfbd3b_19cd830c445.csv to bquxjob_46cfbd3b_19cd830c445.csv
Uploaded: bquxjob_46cfbd3b_19cd830c445.csv


In [4]:
# Load and parse
if filename.endswith('.json'):
    with open(filename) as f:
        raw = json.load(f)
    quidax = pd.DataFrame(raw)
else:
    quidax = pd.read_csv(filename)

quidax['bucket_2h'] = pd.to_datetime(quidax['bucket_2h'], utc=True)
quidax = quidax.set_index('bucket_2h').sort_index()
quidax.index.name = 'datetime'

# Convert numeric columns
numeric_cols = [
    'open', 'high', 'low', 'close', 'volume', 'trade_count',
    'buy_volume', 'sell_volume', 'buy_count', 'sell_count',
    'avg_trade_size', 'max_trade_size', 'stddev_trade_size',
    'large_trade_count', 'large_trade_volume',
    'intrabar_price_stddev', 'intrabar_range',
    'unique_buyers', 'unique_sellers',
    'btcngn_close', 'btcngn_volume', 'btcngn_trade_count',
    'implied_btcusd_quidax',
]
for col in numeric_cols:
    if col in quidax.columns:
        quidax[col] = pd.to_numeric(quidax[col], errors='coerce')

print(f'Rows:       {len(quidax):,}')
print(f'Date range: {quidax.index.min()} to {quidax.index.max()}')
print(f'Days:       {(quidax.index.max() - quidax.index.min()).days}')
print(f'Price range: NGN {quidax["close"].min():,.2f} to NGN {quidax["close"].max():,.2f}')
quidax.head(3)

Rows:       26,252
Date range: 2020-01-08 12:00:00+00:00 to 2026-03-10 04:00:00+00:00
Days:       2252
Price range: NGN 352.53 to NGN 1,889.00


,open,high,low,close,volume,trade_count,buy_volume,sell_volume,buy_count,sell_count,avg_trade_size,max_trade_size,stddev_trade_size,large_trade_count,large_trade_volume,intrabar_price_stddev,intrabar_range,unique_buyers,unique_sellers,btcngn_close,btcngn_volume,btcngn_trade_count,implied_btcusd_quidax
datetime,,,,,,,,,,,,,,,,,,,,,,,
2020-01-08 12:00:00+00:00,362.500000,362.500000,362.500000,362.500000,3.278591,1,0.000000,3.278591,0,1,3.278591,3.278591,NaN,0,0.000000,NaN,0.000000,1,1,3024484.990000,2.181680,49.000000,8343.406869
2020-01-08 14:00:00+00:00,362.440000,362.550000,362.420000,362.550000,89.313612,6,88.929868,0.383744,4,2,14.885602,82.761964,33.322586,0,0.000000,0.067946,0.130000,4,2,3004000.000000,0.873224,40.000000,8285.753689
2020-01-08 16:00:00+00:00,362.550000,362.550000,362.000000,362.000000,4399.324004,19,2595.832762,1803.491243,12,7,231.543369,1437.025644,437.837367,6,4259.253428,0.211544,0.550000,3,2,2905022.290000,3.800651,53.000000,8024.923453


In [5]:
# Data health check
print('=== DATA HEALTH CHECK ===')
print(f'\nNull counts:')
nulls = quidax.isnull().sum()
print(nulls[nulls > 0])

print(f'\nZero-volume bars: {(quidax["volume"] == 0).sum()}')
print(f'Single-trade bars: {(quidax["trade_count"] == 1).sum()}')
print(f'Bars with BTC data: {quidax["btcngn_close"].notna().sum()} / {len(quidax)}')

expected_freq = pd.Timedelta(hours=2)
time_diffs = quidax.index.to_series().diff()
gaps = time_diffs[time_diffs > expected_freq * 1.5]
print(f'\nGaps > 3 hours: {len(gaps)}')
if len(gaps) > 0:
    print(f'Largest gap: {gaps.max()}')

=== DATA HEALTH CHECK ===

Null counts:
stddev_trade_size        563
intrabar_price_stddev    563
btcngn_close               6
btcngn_volume              6
btcngn_trade_count         6
implied_btcusd_quidax      6
dtype: int64

Zero-volume bars: 0
Single-trade bars: 563
Bars with BTC data: 26246 / 26252

Gaps > 3 hours: 495
Largest gap: 0 days 16:00:00


---
## Step 2: Fetch External Data

Sources: Brent, DXY, VIX, USD/ZAR, USD/GHS, USD/KES, BTC/USD, USDNGN official

In [6]:
import yfinance as yf

start_date = quidax.index.min().strftime('%Y-%m-%d')
end_date = (quidax.index.max() + timedelta(days=1)).strftime('%Y-%m-%d')
print(f'Fetching external data: {start_date} to {end_date}\n')

tickers = {
    'BZ=F':      'brent',
    'DX-Y.NYB':  'dxy',
    '^VIX':      'vix',
    'USDZAR=X':  'usdzar',
    'USDGHS=X':  'usdghs',
    'USDKES=X':  'usdkes',
    'BTC-USD':   'btcusd_global',
    'USDNGN=X':  'usdngn_official',
}

external_frames = {}

for ticker, name in tickers.items():
    try:
        data = yf.download(ticker, start=start_date, end=end_date,
                           interval='1d', progress=False, auto_adjust=True)
        if len(data) > 0:
            series = data['Close']
            if isinstance(series, pd.DataFrame):
                series = series.iloc[:, 0]
            external_frames[name] = series.rename(name)
            print(f'  OK {name:20s} {len(data):>6,} daily bars   '
                  f'({series.min():.2f} to {series.max():.2f})')
        else:
            print(f'  -- {name:20s} No data returned')
    except Exception as e:
        print(f'  ERR {name:20s} {e}')

print(f'\nFetched: {len(external_frames)} / {len(tickers)} sources')

Fetching external data: 2020-01-08 to 2026-03-11

  OK brent                 1,553 daily bars   (19.33 to 127.98)
  OK dxy                   1,552 daily bars   (89.44 to 114.11)
  OK vix                   1,550 daily bars   (11.86 to 82.69)
  OK usdzar                1,605 daily bars   (13.42 to 19.77)
  OK usdghs                1,605 daily bars   (5.30 to 573.00)
  OK usdkes                1,605 daily bars   (98.76 to 163.00)
  OK btcusd_global         2,254 daily bars   (4970.79 to 124752.53)
  OK usdngn_official       1,605 daily bars   (359.00 to 1696.20)

Fetched: 8 / 8 sources


In [7]:
# Combine external into one daily DataFrame
external = pd.concat(external_frames.values(), axis=1)
external.index = pd.to_datetime(external.index, utc=True)

print(f'External data: {len(external)} daily rows')
print(f'Date range:    {external.index.min()} to {external.index.max()}')
print(f'\nNull counts:')
print(external.isnull().sum())
external.tail(3)

External data: 2254 daily rows
Date range:    2020-01-08 00:00:00+00:00 to 2026-03-10 00:00:00+00:00

Null counts:
brent              701
dxy                702
vix                704
usdzar             649
usdghs             649
usdkes             649
btcusd_global        0
usdngn_official    649
dtype: int64


,brent,dxy,vix,usdzar,usdghs,usdkes,btcusd_global,usdngn_official
Date,,,,,,,,
2026-03-08 00:00:00+00:00,NaN,NaN,NaN,NaN,NaN,NaN,65969.781250,NaN
2026-03-09 00:00:00+00:00,98.959999,99.180000,25.500000,16.769360,10.741107,127.171059,68402.382812,1387.000000
2026-03-10 00:00:00+00:00,82.339996,98.589996,22.780001,16.175770,10.780000,129.149994,71351.023438,1397.199951


---
## Step 3: Merge Datasets

In [8]:
# Resample external daily to 2-hourly via forward fill
external_2h = external.resample('2h').ffill()

# Merge onto Quidax bars
df = quidax.join(external_2h, how='left')

# Forward fill external columns
ext_cols = list(external_frames.keys())
df[ext_cols] = df[ext_cols].ffill()

print(f'Merged dataset: {len(df):,} rows x {len(df.columns)} columns')
print(f'\nExternal coverage (non-null %):')
for col in ext_cols:
    pct = df[col].notna().mean() * 100
    print(f'  {col:20s} {pct:.1f}%')

df.tail(3)

Merged dataset: 26,252 rows x 31 columns

External coverage (non-null %):
  brent                100.0%
  dxy                  100.0%
  vix                  100.0%
  usdzar               100.0%
  usdghs               100.0%
  usdkes               100.0%
  btcusd_global        100.0%
  usdngn_official      100.0%


,open,high,low,close,volume,trade_count,buy_volume,sell_volume,buy_count,sell_count,avg_trade_size,max_trade_size,stddev_trade_size,large_trade_count,large_trade_volume,intrabar_price_stddev,intrabar_range,unique_buyers,unique_sellers,btcngn_close,btcngn_volume,btcngn_trade_count,implied_btcusd_quidax,brent,dxy,vix,usdzar,usdghs,usdkes,btcusd_global,usdngn_official
datetime,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
2026-03-10 00:00:00+00:00,1420.510000,1426.830000,1418.200000,1418.260000,7294.223000,23,8.059700,7286.163300,3,20,317.140130,1686.216000,521.547947,9,6815.610100,2.138649,8.630000,11,10,98289699.000000,0.038221,20.000000,69303.018487,82.339996,98.589996,22.780001,16.175770,10.780000,129.149994,71351.023438,1397.199951
2026-03-10 02:00:00+00:00,1422.920000,1422.920000,1418.300000,1419.200000,1117.703003,16,200.752003,916.951000,7,9,69.856438,662.697600,162.300128,2,803.256603,1.991689,4.620000,5,3,98341768.000000,0.030713,30.000000,69293.804961,82.339996,98.589996,22.780001,16.175770,10.780000,129.149994,71351.023438,1397.199951
2026-03-10 04:00:00+00:00,1420.480000,1420.500000,1418.410000,1420.500000,208.559700,6,97.921600,110.638100,3,3,34.759950,89.134000,38.004464,0,0.000000,1.099794,2.090000,3,4,98865445.000000,0.011496,10.000000,69599.046111,82.339996,98.589996,22.780001,16.175770,10.780000,129.149994,71351.023438,1397.199951


In [ ]:
# Compute BTC Premium
df['btc_premium'] = (df['implied_btcusd_quidax'] / df['btcusd_global']) - 1

print('BTC Premium stats:')
print(df['btc_premium'].describe())
print(f'\nPremium > 5%: {(df["btc_premium"] > 0.05).sum()} bars')
print(f'Premium < -5%: {(df["btc_premium"] < -0.05).sum()} bars')

---
## Step 4: Feature Engineering

Six categories:
1. Momentum & Trend
2. Cross-Venue Discrepancy
3. Volatility
4. Calendar & Seasonality
5. Cross-Regional African FX
6. Volume dynamics

In [12]:
feat = pd.DataFrame(index=df.index)

# ═══ CATEGORY 1: MOMENTUM & TREND ═══

for periods, label in [(1, '2h'), (4, '8h'), (12, '24h'), (36, '3d'), (84, '7d')]:
    feat[f'return_{label}'] = df['close'].pct_change(periods)

sma_5d = df['close'].rolling(5 * 12).mean()
sma_20d = df['close'].rolling(20 * 12).mean()
feat['sma_5d_20d_cross'] = (sma_5d - sma_20d) / sma_20d.replace(0, np.nan)

ema_12 = df['close'].ewm(span=12).mean()
ema_26 = df['close'].ewm(span=26).mean()
feat['ema_macd'] = (ema_12 - ema_26) / ema_26.replace(0, np.nan)

delta = df['close'].diff()
gain = delta.where(delta > 0, 0).rolling(14).mean()
loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
rs = gain / loss.replace(0, np.nan)
feat['rsi_14'] = 100 - (100 / (1 + rs))

if 'brent' in df.columns:
    feat['brent_roc_3d'] = df['brent'].pct_change(36)
    feat['brent_roc_7d'] = df['brent'].pct_change(84)
if 'dxy' in df.columns:
    feat['dxy_roc_3d'] = df['dxy'].pct_change(36)
    feat['dxy_roc_7d'] = df['dxy'].pct_change(84)

print(f'Category 1 (Momentum): {len(feat.columns)} features')

Category 1 (Momentum): 12 features


In [14]:
# ═══ CATEGORY 2: CROSS-VENUE DISCREPANCY ═══

cat2_start = len(feat.columns)

# BTC premium — compute here if not already done in Step 3
if 'btc_premium' not in df.columns:
    if 'implied_btcusd_quidax' in df.columns and 'btcusd_global' in df.columns:
        df['btc_premium'] = (df['implied_btcusd_quidax'] / df['btcusd_global']) - 1
        print('Computed BTC premium')
    else:
        print('Cannot compute BTC premium — missing columns')

if 'btc_premium' in df.columns:
    feat['btc_premium'] = df['btc_premium']
    feat['btc_premium_ma_12'] = df['btc_premium'].rolling(12).mean()
    feat['btc_premium_std_12'] = df['btc_premium'].rolling(12).std()
    feat['btc_premium_zscore'] = (
        (df['btc_premium'] - df['btc_premium'].rolling(60).mean())
        / df['btc_premium'].rolling(60).std().replace(0, np.nan)
    )

if 'btcusd_global' in df.columns and 'implied_btcusd_quidax' in df.columns:
    feat['quidax_btc_corr_24h'] = (
        df['implied_btcusd_quidax'].rolling(12).corr(df['btcusd_global'])
    )

if 'usdngn_official' in df.columns:
    feat['parallel_vs_official'] = (
        (df['close'] - df['usdngn_official'])
        / df['usdngn_official'].replace(0, np.nan)
    )
    feat['parallel_vs_official_zscore'] = (
        (feat['parallel_vs_official'] - feat['parallel_vs_official'].rolling(360).mean())
        / feat['parallel_vs_official'].rolling(360).std().replace(0, np.nan)
    )
    feat['parallel_vs_official_roc_24h'] = feat['parallel_vs_official'].diff(12)

print(f'Category 2 (Cross-Venue): {len(feat.columns) - cat2_start} features')

Computed BTC premium
Category 2 (Cross-Venue): 8 features


In [15]:
# ═══ CATEGORY 3: VOLATILITY ═══

cat3_start = len(feat.columns)

returns_2h = df['close'].pct_change()
feat['realized_vol_24h'] = returns_2h.rolling(12).std()
feat['realized_vol_7d'] = returns_2h.rolling(84).std()
feat['vol_ratio'] = feat['realized_vol_24h'] / feat['realized_vol_7d'].replace(0, np.nan)

tr = pd.concat([
    df['high'] - df['low'],
    (df['high'] - df['close'].shift(1)).abs(),
    (df['low'] - df['close'].shift(1)).abs(),
], axis=1).max(axis=1)
feat['atr_14'] = tr.rolling(14).mean()
feat['atr_pct'] = feat['atr_14'] / df['close'].replace(0, np.nan)

if 'vix' in df.columns:
    feat['vix'] = df['vix']
    feat['vix_roc_7d'] = df['vix'].pct_change(84)

print(f'Category 3 (Volatility): {len(feat.columns) - cat3_start} features')

Category 3 (Volatility): 7 features


In [16]:
# ═══ CATEGORY 4: CALENDAR & SEASONALITY ═══

cat4_start = len(feat.columns)

feat['hour_sin'] = np.sin(2 * np.pi * df.index.hour / 24)
feat['hour_cos'] = np.cos(2 * np.pi * df.index.hour / 24)
feat['dow_sin'] = np.sin(2 * np.pi * df.index.dayofweek / 7)
feat['dow_cos'] = np.cos(2 * np.pi * df.index.dayofweek / 7)
feat['month_sin'] = np.sin(2 * np.pi * df.index.month / 12)
feat['month_cos'] = np.cos(2 * np.pi * df.index.month / 12)
feat['is_month_end'] = (df.index.day >= 25).astype(float)
feat['is_weekend_adjacent'] = df.index.dayofweek.isin([0, 4]).astype(float)

print(f'Category 4 (Calendar): {len(feat.columns) - cat4_start} features')

Category 4 (Calendar): 8 features


In [17]:
# ═══ CATEGORY 5: CROSS-REGIONAL AFRICAN FX ═══

cat5_start = len(feat.columns)

for col in ['usdzar', 'usdghs', 'usdkes']:
    if col in df.columns:
        feat[f'{col}_roc_1d'] = df[col].pct_change(12)
        feat[f'{col}_roc_7d'] = df[col].pct_change(84)

africa_1d_cols = [c for c in feat.columns if c.endswith('_roc_1d') and c.startswith('usd')]
if africa_1d_cols:
    feat['africa_fx_composite'] = feat[africa_1d_cols].mean(axis=1)

print(f'Category 5 (Regional FX): {len(feat.columns) - cat5_start} features')

Category 5 (Regional FX): 7 features


In [19]:
# ═══ FINALIZE FEATURES ═══

# All features defined above are public. No proprietary features in v2.
public_features = [c for c in feat.columns]

# Replace inf, drop high-null features
feat = feat.replace([np.inf, -np.inf], np.nan)
null_pct = feat.isnull().mean()
drop_cols = null_pct[null_pct > 0.50].index.tolist()
if drop_cols:
    print(f'Dropping {len(drop_cols)} high-null features: {drop_cols}')
    feat = feat.drop(columns=drop_cols)

public_features = [c for c in feat.columns]

print(f'\nTotal features: {len(public_features)}')
print(f'Feature list: {public_features}')


Total features: 42
Feature list: ['return_2h', 'return_8h', 'return_24h', 'return_3d', 'return_7d', 'sma_5d_20d_cross', 'ema_macd', 'rsi_14', 'brent_roc_3d', 'brent_roc_7d', 'dxy_roc_3d', 'dxy_roc_7d', 'btc_premium', 'btc_premium_ma_12', 'btc_premium_std_12', 'btc_premium_zscore', 'quidax_btc_corr_24h', 'parallel_vs_official', 'parallel_vs_official_zscore', 'parallel_vs_official_roc_24h', 'realized_vol_24h', 'realized_vol_7d', 'vol_ratio', 'atr_14', 'atr_pct', 'vix', 'vix_roc_7d', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'month_sin', 'month_cos', 'is_month_end', 'is_weekend_adjacent', 'usdzar_roc_1d', 'usdzar_roc_7d', 'usdghs_roc_1d', 'usdghs_roc_7d', 'usdkes_roc_1d', 'usdkes_roc_7d', 'africa_fx_composite']


---
## Step 5: Target Variable & Recency Weights

**Blended target:** where USDNGN official rate exists, blend 70% Quidax + 30% official to reduce circularity.

**Recency weighting:** exponential decay (half-life 180 days) so the model prioritizes current market dynamics over 2020-2021 devaluation patterns.

In [36]:
# ═══ TARGET VARIABLE ═══
# The Quidax USDT/NGN close IS the rate the OTC desk trades at.
# Predicting its next 2-hour move is exactly the right objective.
# USDNGN official rate remains a FEATURE (cross-venue signal), not part of the target.

feat['target'] = df['close'].pct_change(1).shift(-1)
target_type = 'Quidax USDT/NGN close (OTC execution price)'

feat = feat.dropna(subset=['target'])
feat[public_features] = feat[public_features].ffill().fillna(0)

print(f'Target: {target_type}')
print(f'Usable rows: {len(feat):,}')
print(f'Target stats: mean={feat["target"].mean()*100:.4f}%, std={feat["target"].std()*100:.4f}%')

Target: Quidax USDT/NGN close (OTC execution price)
Usable rows: 26,251
Target stats: mean=0.0134%, std=1.3322%


In [38]:
# ═══ RECENCY WEIGHTS ═══

def compute_sample_weights(index, half_life_days=180):
    """Exponential decay: most recent row = 1.0, oldest ~ 0.05."""
    days_ago = (index.max() - index).total_seconds() / 86400
    weights = np.exp(-np.log(2) * days_ago / half_life_days)
    weights = np.clip(weights, 0.05, 1.0)
    return weights

sample_weights = compute_sample_weights(feat.index, half_life_days=180)

print(f'Weight at most recent row: {sample_weights[-1]:.3f}')
print(f'Weight at 6 months ago:    ~{sample_weights[len(sample_weights)//2]:.3f}')
print(f'Weight at oldest row:      {sample_weights[0]:.3f}')

Weight at most recent row: 1.000
Weight at 6 months ago:    ~0.050
Weight at oldest row:      0.050


---
## Step 6: Walk-Forward Cross Validation

In [39]:
X = feat[public_features]
y = feat['target']

COST_BPS = 5.0
PRODUCTION_THRESHOLD = 0.0030

def purged_walk_forward(n_rows, n_splits=5, train_pct=0.60, embargo=12):
    test_size = int(n_rows * (1 - train_pct) / n_splits)
    splits = []
    for i in range(n_splits):
        test_start = int(n_rows * train_pct) + i * test_size
        test_end = min(test_start + test_size, n_rows)
        train_end = test_start - embargo
        if train_end > 0 and test_end <= n_rows:
            splits.append((slice(0, train_end), slice(test_start, test_end)))
    return splits

splits = purged_walk_forward(len(X), n_splits=5, embargo=12)
print(f'Walk-forward splits: {len(splits)}')
print(f'Embargo: 12 bars (24 hours)')
print(f'Production threshold: {PRODUCTION_THRESHOLD*100:.2f}%\n')

cv_results = []

for fold, (train_sl, test_sl) in enumerate(splits):
    X_train_cv, y_train_cv = X.iloc[train_sl], y.iloc[train_sl]
    X_test_cv, y_test_cv = X.iloc[test_sl], y.iloc[test_sl]
    w_train_cv = sample_weights[train_sl]

    scaler_cv = StandardScaler()
    Xtr_cv = scaler_cv.fit_transform(X_train_cv)
    Xte_cv = scaler_cv.transform(X_test_cv)

    xgb_cv = XGBRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                          subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0,
                          random_state=42, verbosity=0)
    xgb_cv.fit(Xtr_cv, y_train_cv, sample_weight=w_train_cv)

    lgbm_cv = LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                            subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0,
                            random_state=42, verbosity=-1)
    lgbm_cv.fit(Xtr_cv, y_train_cv, sample_weight=w_train_cv)

    ridge_cv = Ridge(alpha=1.0)
    ridge_cv.fit(Xtr_cv, y_train_cv, sample_weight=w_train_cv)

    pred_cv = (0.50 * xgb_cv.predict(Xte_cv) +
               0.30 * lgbm_cv.predict(Xte_cv) +
               0.20 * ridge_cv.predict(Xte_cv))
    actual_cv = y_test_cv.values

    dir_correct_cv = np.sign(pred_cv) == np.sign(actual_cv)
    dir_acc_cv = dir_correct_cv.mean() * 100

    cost = COST_BPS / 10000
    pnl_cv = np.where(dir_correct_cv, np.abs(actual_cv) - cost, -np.abs(actual_cv) - cost)
    mask_cv = np.abs(pred_cv) > PRODUCTION_THRESHOLD
    active_pnl_cv = pnl_cv[mask_cv].sum() * 10000
    n_trades_cv = mask_cv.sum()

    # Directional accuracy at production threshold
    if mask_cv.sum() > 0:
        dir_acc_at_thresh = dir_correct_cv[mask_cv].mean() * 100
    else:
        dir_acc_at_thresh = 0

    mae_cv = mean_absolute_error(actual_cv, pred_cv)

    cv_results.append({
        'fold': fold + 1,
        'train_rows': X_train_cv.shape[0],
        'test_rows': X_test_cv.shape[0],
        'dir_acc_%': round(dir_acc_cv, 2),
        'dir_acc_at_30bps_%': round(dir_acc_at_thresh, 2),
        'mae': round(mae_cv, 6),
        'net_pnl_bps': round(active_pnl_cv, 2),
        'trades': int(n_trades_cv),
        'test_start': str(X_test_cv.index[0].date()),
        'test_end': str(X_test_cv.index[-1].date()),
    })

    print(f'Fold {fold+1}: DirAcc={dir_acc_cv:.1f}% | @30bps={dir_acc_at_thresh:.1f}% | '
          f'PnL={active_pnl_cv:+.0f}bps | Trades={n_trades_cv} | '
          f'{X_test_cv.index[0].date()} to {X_test_cv.index[-1].date()}')

cv_df = pd.DataFrame(cv_results)
print(f'\n{"=" * 60}')
print(f'CV SUMMARY (recency-weighted)')
print(f'Mean DirAcc (all):    {cv_df["dir_acc_%"].mean():.1f}%')
print(f'Mean DirAcc (@30bps): {cv_df["dir_acc_at_30bps_%"].mean():.1f}%')
print(f'Mean PnL:             {cv_df["net_pnl_bps"].mean():+.0f} bps')
print(f'Mean MAE:             {cv_df["mae"].mean():.6f}')
print(f'Last fold PnL:        {cv_df["net_pnl_bps"].iloc[-1]:+.0f} bps')

Walk-forward splits: 5
Embargo: 12 bars (24 hours)
Production threshold: 0.30%

Fold 1: DirAcc=55.7% | @30bps=62.5% | PnL=+34545bps | Trades=811 | 2023-10-16 to 2024-04-08
Fold 2: DirAcc=61.2% | @30bps=63.7% | PnL=+6385bps | Trades=466 | 2024-04-08 to 2024-09-30
Fold 3: DirAcc=56.0% | @30bps=67.7% | PnL=+1613bps | Trades=96 | 2024-09-30 to 2025-03-25
Fold 4: DirAcc=58.4% | @30bps=50.0% | PnL=+1130bps | Trades=22 | 2025-03-25 to 2025-09-16
Fold 5: DirAcc=56.0% | @30bps=72.7% | PnL=+112bps | Trades=11 | 2025-09-16 to 2026-03-10

CV SUMMARY (recency-weighted)
Mean DirAcc (all):    57.5%
Mean DirAcc (@30bps): 63.3%
Mean PnL:             +8757 bps
Mean MAE:             0.003155
Last fold PnL:        +112 bps


---
## Step 7: Optuna-Tuned Final Model

In [41]:
# ═══ SPLIT: 70% train, 15% val, 15% holdout ═══

split_train = int(len(X) * 0.70)
split_val = int(len(X) * 0.85)

X_train = X.iloc[:split_train]
y_train = y.iloc[:split_train]
w_train = sample_weights[:split_train]

X_val = X.iloc[split_train:split_val]
y_val = y.iloc[split_train:split_val]
w_val = sample_weights[split_train:split_val]

X_holdout = X.iloc[split_val:]
y_holdout = y.iloc[split_val:]

scaler_tune = StandardScaler()
Xtr = scaler_tune.fit_transform(X_train)
Xva = scaler_tune.transform(X_val)
Xho = scaler_tune.transform(X_holdout)

print(f'Train:    {len(X_train):,} rows  ({X_train.index.min().date()} to {X_train.index.max().date()})')
print(f'Val:      {len(X_val):,} rows  ({X_val.index.min().date()} to {X_val.index.max().date()})')
print(f'Holdout:  {len(X_holdout):,} rows  ({X_holdout.index.min().date()} to {X_holdout.index.max().date()})')
print(f'Target:   {target_type}')

Train:    18,375 rows  (2020-01-08 to 2024-05-22)
Val:      3,938 rows  (2024-05-22 to 2025-04-15)
Holdout:  3,938 rows  (2025-04-16 to 2026-03-10)
Target:   Quidax USDT/NGN close (OTC execution price)


In [42]:
# ═══ OPTUNA TUNING (optimizes at production threshold) ═══

def objective(trial):
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 100, 500),
        'max_depth': trial.suggest_int('max_depth', 3, 8),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        'subsample': trial.suggest_float('subsample', 0.6, 0.9),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 0.8),
        'reg_lambda': trial.suggest_float('reg_lambda', 1.0, 5.0),
        'min_child_weight': trial.suggest_int('min_child_weight', 5, 50),
        'random_state': 42,
        'verbosity': 0,
    }
    model = XGBRegressor(**params)
    model.fit(Xtr, y_train, sample_weight=w_train)
    pred = model.predict(Xva)
    correct = np.sign(pred) == np.sign(y_val.values)
    cost = COST_BPS / 10000
    pnl = np.where(correct, np.abs(y_val.values) - cost, -np.abs(y_val.values) - cost)
    mask = np.abs(pred) > PRODUCTION_THRESHOLD
    return pnl[mask].sum() * 10000 if mask.sum() > 10 else -9999

print('Tuning XGBoost with recency weights (50 trials)...')
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, show_progress_bar=True)

print(f'\nBest val PnL: {study.best_value:+.1f} bps')
print(f'Best params: {study.best_params}')

Tuning XGBoost with recency weights (50 trials)...


  0%|          | 0/50 [00:00<?, ?it/s]


Best val PnL: +4785.6 bps
Best params: {'n_estimators': 491, 'max_depth': 7, 'learning_rate': 0.0338299751135883, 'subsample': 0.6073434124051657, 'colsample_bytree': 0.7096858517889113, 'reg_lambda': 2.3488244901159065, 'min_child_weight': 50}


In [43]:
# ═══ TRAIN FINAL ENSEMBLE ON TRAIN+VAL ═══

best_params = study.best_params
best_params['random_state'] = 42
best_params['verbosity'] = 0

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])
w_trainval = np.concatenate([w_train, w_val])

final_scaler = StandardScaler()
Xtv = final_scaler.fit_transform(X_trainval)
Xho_final = final_scaler.transform(X_holdout)

final_xgb = XGBRegressor(**best_params)
final_xgb.fit(Xtv, y_trainval, sample_weight=w_trainval)

final_lgbm = LGBMRegressor(n_estimators=200, max_depth=5, learning_rate=0.05,
                           subsample=0.8, colsample_bytree=0.7, reg_lambda=2.0,
                           random_state=42, verbosity=-1)
final_lgbm.fit(Xtv, y_trainval, sample_weight=w_trainval)

final_ridge = Ridge(alpha=1.0)
final_ridge.fit(Xtv, y_trainval, sample_weight=w_trainval)

print('Final ensemble trained on train+val with recency weights.')

Final ensemble trained on train+val with recency weights.


---
## Step 8: Holdout Evaluation & Threshold Sweep

In [44]:
# ═══ HOLDOUT PREDICTIONS ═══

pred_ho = (0.50 * final_xgb.predict(Xho_final) +
           0.30 * final_lgbm.predict(Xho_final) +
           0.20 * final_ridge.predict(Xho_final))

actual_ho = y_holdout.values
dir_correct = np.sign(pred_ho) == np.sign(actual_ho)
dir_acc = dir_correct.mean() * 100

cost = COST_BPS / 10000
pnl_arr = np.where(dir_correct, np.abs(actual_ho) - cost, -np.abs(actual_ho) - cost)

mask_prod = np.abs(pred_ho) > PRODUCTION_THRESHOLD
net_pnl = pnl_arr[mask_prod].sum() * 10000
n_trades = mask_prod.sum()
mae = mean_absolute_error(actual_ho, pred_ho)

if mask_prod.sum() > 0:
    dir_acc_at_thresh = dir_correct[mask_prod].mean() * 100
else:
    dir_acc_at_thresh = 0

print(f'{"=" * 60}')
print(f'RECENCY-WEIGHTED MODEL — HOLDOUT RESULTS')
print(f'{"=" * 60}')
print(f'Target:             {target_type}')
print(f'Dir Accuracy (all): {dir_acc:.1f}%')
print(f'Dir Accuracy @30bps:{dir_acc_at_thresh:.1f}%')
print(f'Net PnL @30bps:     {net_pnl:+.1f} bps')
print(f'Trades @30bps:      {n_trades}')
print(f'MAE:                {mae:.6f}')
print(f'Features:           {len(public_features)}')

RECENCY-WEIGHTED MODEL — HOLDOUT RESULTS
Target:             Quidax USDT/NGN close (OTC execution price)
Dir Accuracy (all): 57.3%
Dir Accuracy @30bps:76.9%
Net PnL @30bps:     +1796.8 bps
Trades @30bps:      26
MAE:                0.001274
Features:           42


In [45]:
# ═══ THRESHOLD SWEEP ═══

print(f'{"Threshold":>12} {"DirAcc":>8} {"Trades":>8} {"PnL(bps)":>10} {"PnL/Trade":>10}')
print('-' * 52)

for thresh in [0.0010, 0.0015, 0.0020, 0.0030, 0.0050, 0.0075, 0.0100]:
    mask = np.abs(pred_ho) > thresh
    if mask.sum() == 0:
        continue
    dc = dir_correct[mask]
    acc = dc.mean() * 100
    p = pnl_arr[mask].sum() * 10000
    pt = p / mask.sum()
    marker = '  <<<' if thresh == PRODUCTION_THRESHOLD else ''
    print(f'{thresh*100:>11.2f}% {acc:>7.1f}% {mask.sum():>8} {p:>+10.1f} {pt:>+10.1f}{marker}')

   Threshold   DirAcc   Trades   PnL(bps)  PnL/Trade
----------------------------------------------------
       0.10%    70.4%      544    +3820.1       +7.0
       0.15%    68.6%      191    +2694.5      +14.1
       0.20%    67.6%       74    +1927.7      +26.0
       0.30%    76.9%       26    +1796.8      +69.1  <<<
       0.50%    70.0%       10    +1595.0     +159.5
       0.75%    71.4%        7    +1529.9     +218.6
       1.00%    75.0%        4    +1345.3     +336.3


---
## Step 9: Feature Importance

In [46]:
import plotly.graph_objects as go

fi = pd.Series(final_xgb.feature_importances_, index=public_features).sort_values(ascending=True)
top = fi.tail(25)

fig = go.Figure(go.Bar(x=top.values, y=top.index, orientation='h',
                       marker_color='#3b82f6'))
fig.update_layout(
    title='Top 25 Features (Tuned Recency-Weighted Model)',
    height=700, template='plotly_dark',
    margin=dict(l=250, r=20, t=50, b=20),
)
fig.show()

fi_full = pd.Series(final_xgb.feature_importances_, index=public_features)
categories = {
    'Momentum':     [c for c in public_features if any(c.startswith(p) for p in ['return_', 'sma_', 'ema_', 'rsi_', 'brent_roc', 'dxy_roc'])],
    'Cross-Venue':  [c for c in public_features if any(c.startswith(p) for p in ['btc_premium', 'quidax_btc', 'parallel_vs'])],
    'Regional FX':  [c for c in public_features if any(c.startswith(p) for p in ['usdzar', 'usdghs', 'usdkes', 'africa_'])],
    'Volatility':   [c for c in public_features if any(c.startswith(p) for p in ['realized_', 'vol_ratio', 'atr_', 'vix'])],
    'Calendar':     [c for c in public_features if any(c.startswith(p) for p in ['hour_', 'dow_', 'month_', 'is_'])],
}
print('\nImportance by category:')
for cat_name, cols in categories.items():
    cat_imp = fi_full[fi_full.index.isin(cols)].sum()
    print(f'  {cat_name:15s} {cat_imp:.4f}  ({cat_imp / fi_full.sum() * 100:.1f}%)')


Importance by category:
  Momentum        0.3930  (39.3%)
  Cross-Venue     0.1871  (18.7%)
  Regional FX     0.0816  (8.2%)
  Volatility      0.2749  (27.5%)
  Calendar        0.0633  (6.3%)


---
## Step 10: Export Production Artifacts

In [49]:
import pickle
import json as json_lib

# Save models
for name, model in [('xgb', final_xgb), ('lgbm', final_lgbm), ('ridge', final_ridge)]:
    with open(f'{name}_model.pkl', 'wb') as f:
        pickle.dump(model, f)
    print(f'Saved {name}_model.pkl')

# Save scaler
with open('scaler.pkl', 'wb') as f:
    pickle.dump(final_scaler, f)
print('Saved scaler.pkl')

# Save feature list
with open('feature_cols.json', 'w') as f:
    json_lib.dump(public_features, f, indent=2)
print(f'Saved feature_cols.json ({len(public_features)} features)')

# Save metadata
metadata = {
    'trained_at': datetime.now().isoformat(),
    'model_version': 'v2_public_recency_weighted',
    'target_type': target_type,
    'training_rows': len(X_trainval),
    'holdout_rows': len(X_holdout),
    'n_features': len(public_features),
    'holdout_dir_acc': round(dir_acc, 2),
    'holdout_dir_acc_at_30bps': round(dir_acc_at_thresh, 2),
    'holdout_net_pnl_bps': round(net_pnl, 2),
    'holdout_trades_at_30bps': int(n_trades),
    'holdout_mae': round(mae, 6),
    'recommended_threshold': PRODUCTION_THRESHOLD,
    'recency_half_life_days': 180,
    'xgb_params': study.best_params,
    'data_start': str(feat.index.min()),
    'data_end': str(feat.index.max()),
}
with open('model_metadata.json', 'w') as f:
    json_lib.dump(metadata, f, indent=2)

print('\nSaved model_metadata.json')
print(json_lib.dumps(metadata, indent=2))

Saved xgb_model.pkl
Saved lgbm_model.pkl
Saved ridge_model.pkl
Saved scaler.pkl
Saved feature_cols.json (42 features)

Saved model_metadata.json
{
  "trained_at": "2026-03-10T18:15:12.414529",
  "model_version": "v2_public_recency_weighted",
  "target_type": "Quidax USDT/NGN close (OTC execution price)",
  "training_rows": 22313,
  "holdout_rows": 3938,
  "n_features": 42,
  "holdout_dir_acc": 57.26,
  "holdout_dir_acc_at_30bps": 76.92,
  "holdout_net_pnl_bps": 1796.77,
  "holdout_trades_at_30bps": 26,
  "holdout_mae": 0.001274,
  "recommended_threshold": 0.003,
  "recency_half_life_days": 180,
  "xgb_params": {
    "n_estimators": 491,
    "max_depth": 7,
    "learning_rate": 0.0338299751135883,
    "subsample": 0.6073434124051657,
    "colsample_bytree": 0.7096858517889113,
    "reg_lambda": 2.3488244901159065,
    "min_child_weight": 50
  },
  "data_start": "2020-01-08 12:00:00+00:00",
  "data_end": "2026-03-10 02:00:00+00:00"
}


In [50]:
# Save CV results and feature importance
cv_df.to_csv('cv_results.csv', index=False)
print('Saved cv_results.csv')

fi_export = pd.Series(final_xgb.feature_importances_, index=public_features).sort_values(ascending=False)
fi_export.to_csv('feature_importance.csv')
print('Saved feature_importance.csv')

# Download all
from google.colab import files
for f in ['xgb_model.pkl', 'lgbm_model.pkl', 'ridge_model.pkl',
          'scaler.pkl', 'feature_cols.json', 'model_metadata.json',
          'cv_results.csv', 'feature_importance.csv']:
    files.download(f)
    print(f'Downloaded {f}')

Saved cv_results.csv
Saved feature_importance.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded xgb_model.pkl


<IPython.core.display.Javascript object>

...

<IPython.core.display.Javascript object>

Downloaded cv_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Downloaded feature_importance.csv


---
## Step 11: Training Report

In [53]:
print('=' * 60)
print('USD/NGN 2-HOUR PREDICTION — TRAINING REPORT v2')
print('=' * 60)

fi_sorted = pd.Series(final_xgb.feature_importances_, index=public_features).sort_values(ascending=False)
top5 = list(fi_sorted.head(5).index)

print(f'''
DATA
  Source:           Quidax USDT/NGN exchange + Yahoo Finance
  Bars:             {len(feat):,} (2-hour intervals)
  Period:           {feat.index.min().date()} to {feat.index.max().date()}
  Features:         {len(public_features)}
  Target:           {target_type}
  Recency weight:   180-day half-life

CROSS-VALIDATION (5-fold purged walk-forward, 24h embargo)
  Mean DirAcc (all):    {cv_df["dir_acc_%"].mean():.1f}%
  Mean DirAcc (@30bps): {cv_df["dir_acc_at_30bps_%"].mean():.1f}%
  Mean Net PnL:         {cv_df["net_pnl_bps"].mean():+.0f} bps
  Last fold PnL:        {cv_df["net_pnl_bps"].iloc[-1]:+.0f} bps

HOLDOUT (last 15%: {X_holdout.index.min().date()} to {X_holdout.index.max().date()})
  Dir Accuracy (all):   {dir_acc:.1f}%
  Dir Accuracy (@30bps):{dir_acc_at_thresh:.1f}%
  Net PnL (@30bps):     {net_pnl:+.1f} bps
  Trades (@30bps):      {n_trades}
  MAE:                  {mae:.6f}

TOP 5 DRIVERS
  1. {top5[0]}
  2. {top5[1]}
  3. {top5[2]}
  4. {top5[3]}
  5. {top5[4]}

ARTIFACTS EXPORTED
  xgb_model.pkl, lgbm_model.pkl, ridge_model.pkl
  scaler.pkl, feature_cols.json, model_metadata.json
  cv_results.csv, feature_importance.csv
''')

if dir_acc > 52 and net_pnl > 0:
    print('RESULT: MODEL PASSES — Proceed to build live inference system.')
else:
    print('RESULT: Below threshold. Investigate further.')

USD/NGN 2-HOUR PREDICTION — TRAINING REPORT v2

DATA
  Source:           Quidax USDT/NGN exchange + Yahoo Finance
  Bars:             26,251 (2-hour intervals)
  Period:           2020-01-08 to 2026-03-10
  Features:         42
  Target:           Quidax USDT/NGN close (OTC execution price)
  Recency weight:   180-day half-life

CROSS-VALIDATION (5-fold purged walk-forward, 24h embargo)
  Mean DirAcc (all):    57.5%
  Mean DirAcc (@30bps): 63.3%
  Mean Net PnL:         +8757 bps
  Last fold PnL:        +112 bps

HOLDOUT (last 15%: 2025-04-16 to 2026-03-10)
  Dir Accuracy (all):   57.3%
  Dir Accuracy (@30bps):76.9%
  Net PnL (@30bps):     +1796.8 bps
  Trades (@30bps):      26
  MAE:                  0.001274

TOP 5 DRIVERS
  1. return_2h
  2. return_8h
  3. realized_vol_24h
  4. realized_vol_7d
  5. btc_premium

ARTIFACTS EXPORTED
  xgb_model.pkl, lgbm_model.pkl, ridge_model.pkl
  scaler.pkl, feature_cols.json, model_metadata.json
  cv_results.csv, feature_importance.csv

RESULT: MODE